# UAV Battery Tool — Notebook 05: Report Generation

Generate formatted Excel reports from simulation results, battery comparisons, and flight log analysis.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import warnings; warnings.filterwarnings('ignore')

from batteries.database import BatteryDatabase
from batteries.log_importer import generate_synthetic_log
from batteries.parameter_fitter import fit_all, apply_fitted_params
from mission.simulator import run_simulation, compare_batteries, temperature_sweep
from mission.report_generator import generate_report

DB_PATH = '../battery_db.xlsx'
db = BatteryDatabase(DB_PATH).load()
print(db.summary())

═══ Battery Database Summary ═══
  Chemistries       : 9
  Cells             : 11
  Battery packs     : 8
  Discharge points  : 132
  Equipment items   : 29
  UAV configurations: 3
  Mission profiles  : 3


## 1 · Configure Report Parameters

In [2]:
# ── Select packs, mission, and UAV ────────────────────────────────────────────
PRIMARY_PACK_ID    = 'BAT_MID_6S2P'
COMPARE_PACK_IDS   = ['BAT_MID_6S2P', 'BAT_MID_6S4P', 'BAT_AG_6S1P']
MISSION_ID         = 'SURVEY_STD'
UAV_ID             = 'HEX_SURVEY_900'
AMBIENT_TEMP_C     = 25.0
TEMP_SWEEP         = [-20,-15,-10,-5,0,5,10,15,20,25,30,35,40,45]
INCLUDE_LOG        = True   # generate synthetic log and fit parameters
OUT_PATH           = 'UAV_Battery_Report.xlsx'
# ─────────────────────────────────────────────────────────────────────────────

primary_pack = db.packs[PRIMARY_PACK_ID]
mission      = db.missions[MISSION_ID]
uav          = db.uav_configs[UAV_ID]
compare_packs= [db.packs[pid] for pid in COMPARE_PACK_IDS if pid in db.packs]
print(f'Primary pack  : {primary_pack}')
print(f'Compare packs : {[p.battery_id for p in compare_packs]}')
print(f'Mission       : {mission}')

Primary pack  : BAT_MID_6S2P: Mid UAV 6S2P 21700 | 6S2P | 21.9V 9.0Ah (197Wh) | 840g
Compare packs : ['BAT_MID_6S2P', 'BAT_MID_6S4P', 'BAT_AG_6S1P']
Mission       : Mission 'Grid Survey 500m' (SURVEY_STD): 8 phases | 24.5 min | 6700m


## 2 · Run Simulations

In [3]:
print('Running simulations...')

# Primary simulation
primary_result = run_simulation(
    pack=primary_pack, mission=mission, uav=uav,
    discharge_pts=db.discharge_pts,
    ambient_temp_c=AMBIENT_TEMP_C, dt_s=1.0
)
print(primary_result.summary())

# Multi-pack comparison
compare_results = compare_batteries(
    packs=compare_packs, mission=mission, uav=uav,
    discharge_pts=db.discharge_pts,
    ambient_temp_c=AMBIENT_TEMP_C, dt_s=2.0
)
print(f'\nMulti-pack results:')
for r in compare_results:
    print(f'  {r.pack_id}: SoC={r.final_soc:.1f}%  Vmin={r.min_voltage:.3f}V  depleted={r.depleted}')

Running simulations...
════════════════════════════════════════════════════
 Simulation: BAT_MID_6S2P × SURVEY_STD  [COMPLETED]
════════════════════════════════════════════════════
  Duration         : 1469 s  (24.5 min)
  Energy consumed  : 127.91 Wh
  Initial SoC      : 100.0 %
  Final SoC        : 34.0 %
  Min voltage      : 19.657 V
  Max current      : 30.7 A
  Peak sag total   : 1.537 V
  Peak temperature : 30.7 °C

Multi-pack results:
  BAT_MID_6S2P: SoC=34.0%  Vmin=19.661V  depleted=False
  BAT_MID_6S4P: SoC=71.2%  Vmin=19.257V  depleted=False
  BAT_AG_6S1P: SoC=33.5%  Vmin=19.113V  depleted=False


In [4]:
# Temperature sweep
print(f'Running temperature sweep ({len(TEMP_SWEEP)} temperatures)...')
sweep_results = temperature_sweep(
    pack=primary_pack, mission=mission, uav=uav,
    discharge_pts=db.discharge_pts,
    temperatures_c=TEMP_SWEEP, dt_s=5.0
)
print('Done.')

df_sweep = pd.DataFrame([{
    'Temp (C)': t, 'Final SoC (%)': round(r.final_soc,1),
    'Peak sag (V)': round(r.peak_sag_v,3), 'Min V (V)': round(r.min_voltage,3),
    'Max T (C)': round(r.max_temp_c,1), 'Depleted': r.depleted}
    for t, r in zip(TEMP_SWEEP, sweep_results)])
print(df_sweep.to_string(index=False))

Running temperature sweep (14 temperatures)...
Done.
 Temp (C)  Final SoC (%)  Peak sag (V)  Min V (V)  Max T (C)  Depleted
      -20           10.0        14.529     11.428        0.4      True
      -15           11.4         7.722     16.401        2.3     False
      -10           16.2         5.672     17.325        4.9     False
       -5           20.4         4.422     17.903        7.9     False
        0           24.3         3.559     18.367       11.1     False
        5           27.2         2.926     18.740       14.7     False
       10           29.7         2.446     19.056       18.4     False
       15           31.9         2.072     19.332       22.4     False
       20           33.5         1.776     19.555       26.5     False
       25           34.0         1.537     19.674       30.7     False
       30           33.4         1.342     19.702       35.1     False
       35           32.5         1.181     19.699       39.6     False
       40           31.3

## 3 · Flight Log Analysis (optional)

In [5]:
log = fitted = None
if INCLUDE_LOG:
    from batteries.log_importer import generate_synthetic_log
    from batteries.parameter_fitter import fit_all, apply_fitted_params
    print('Generating synthetic flight log and fitting parameters...')
    log = generate_synthetic_log(primary_pack, mission, uav, db.discharge_pts,
                                  ambient_temp_c=AMBIENT_TEMP_C, dt_s=2.0,
                                  noise_v=0.03, noise_i=0.8)
    fitted = fit_all(log, primary_pack.pack_capacity_ah,
                     primary_pack.pack_voltage_cutoff,
                     chem_id=primary_pack.chemistry_id,
                     pack_id=primary_pack.battery_id)
    print(fitted.summary())
else:
    print('Skipping log analysis (INCLUDE_LOG=False)')

Generating synthetic flight log and fitting parameters...
  [1/5] Fitting internal resistance...
        R_internal_mohm: 48.3790 ± 14.9090  R²=-0.018  n=716
  [2/5] Reconstructing OCV curve...
        11 OCV points  R²=0.064
  [3/5] Fitting Peukert exponent...
        peukert_k: 1.0000 ± 0.0200  R²=0.000  n=733
  [4/5] Fitting Arrhenius temperature coefficients...
        B_ohmic_K: 0.0000 ± 0.0000  R²=0.000  n=0
        B_ct_K: 0.0000 ± 0.0000  R²=0.000  n=0
  [5/5] Estimating actual capacity...
        actual_capacity_ah: 9.0000 ± 0.0000  R²=0.000  n=0
══ Fitted Battery Parameters ══
  R_internal   : R_internal_mohm: 48.3790 ± 14.9090  R²=-0.018  n=716
  Capacity     : actual_capacity_ah: 9.0000 ± 0.0000  R²=0.000  n=0
  Peukert k    : peukert_k: 1.0000 ± 0.0200  R²=0.000  n=733
  B_ohmic      : B_ohmic_K: 0.0000 ± 0.0000  R²=0.000  n=0
  B_ct         : B_ct_K: 0.0000 ± 0.0000  R²=0.000  n=0
  OCV curve    : 11 points fitted
  ⚠  IR regression R²=-0.02 — low confidence (<0.5). Consi

## 4 · Generate Excel Report

In [6]:
print(f'Generating report: {OUT_PATH}')
report_path = generate_report(
    out_path=OUT_PATH,
    results=compare_results,
    packs=compare_packs,
    mission=mission,
    uav_name=UAV_ID,
    ambient_temp_c=AMBIENT_TEMP_C,
    temp_sweep_temps=TEMP_SWEEP,
    temp_sweep_results=sweep_results,
    flight_log=log,
    fitted_params=fitted,
    primary_pack=primary_pack,
)
print(f'Report saved: {report_path}')

Generating report: UAV_Battery_Report.xlsx
✓ Report saved → UAV_Battery_Report.xlsx
Report saved: UAV_Battery_Report.xlsx


## 5 · Quick Battery Selection Table

Which batteries from the catalog can safely complete this mission at all temperature sweep points?

In [7]:
from mission.simulator import SimulationResult
all_packs = list(db.packs.values())
selection_rows = []
print(f'Testing {len(all_packs)} packs across {len(TEMP_SWEEP)} temperatures...')
for p in all_packs:
    row = {'Pack_ID': p.battery_id, 'Chemistry': p.chemistry_id,
           'Energy (Wh)': p.pack_energy_wh, 'Weight (g)': p.pack_weight_g,
           'Wh/kg': p.specific_energy_wh_kg}
    pass_all = True
    for temp in [-10, 0, 15, 25, 35]:
        try:
            r = run_simulation(p, mission, uav, db.discharge_pts,
                               ambient_temp_c=temp, dt_s=10.0)
            row[f'{temp}C SoC'] = round(r.final_soc, 1)
            row[f'{temp}C pass'] = not r.depleted and r.final_soc > 15
            if r.depleted or r.final_soc <= 15: pass_all = False
        except Exception as e:
            row[f'{temp}C SoC'] = 'ERR'; row[f'{temp}C pass'] = False
            pass_all = False
    row['All temps PASS'] = pass_all
    selection_rows.append(row)

df_sel = pd.DataFrame(selection_rows).set_index('Pack_ID')
df_sel.sort_values('All temps PASS', ascending=False)

Testing 8 packs across 14 temperatures...


,Chemistry,Energy (Wh),Weight (g),Wh/kg,-10C SoC,-10C pass,0C SoC,0C pass,15C SoC,15C pass,25C SoC,25C pass,35C SoC,35C pass,All temps PASS
Pack_ID,,,,,,,,,,,,,,,
BAT_MID_6S2P,LION21,197.00,840.0,235.0,16.2,True,24.3,True,31.9,True,34.0,True,32.5,True,True
BAT_LE_12S2P,LION21,350.00,1656.0,212.0,54.3,True,60.1,True,64.9,True,66.6,True,66.0,True,True
BAT_FPV_4S1P,LIPO,19.20,128.0,150.0,10.0,False,10.0,False,10.0,False,10.0,False,10.0,False,False
BAT_MICRO_1S,LIHV,2.12,14.0,151.0,10.0,False,10.0,False,10.0,False,10.0,False,10.0,False,False
BAT_MID_6S4P,LION21,436.00,1692.0,258.0,10.0,False,10.0,False,70.3,True,71.2,True,71.0,True,False
BAT_AG_6S1P,LIFEPO4,192.00,1860.0,103.0,10.0,False,18.0,True,30.0,True,33.5,True,33.0,True,False
BAT_HLIFT_6S2P,LIFEPO4,99.00,912.0,109.0,10.0,False,10.0,False,10.0,False,10.0,False,10.0,False,False
BAT_SSS_6S1P,SSS,111.00,606.0,183.0,10.0,False,10.0,False,10.0,False,10.0,False,10.0,False,False
